In [2]:
%pip install numpy

   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
    --------------------------------------- 0.3/12.4 MB ? eta -:--:--
    --------------------------------------- 0.3/12.4 MB ? eta -:--:--
    --------------------------------------- 0.3/12.4 MB ? eta -:--:--
    --------------------------------------- 0.3/12.4 MB ? eta -:--:--
    --------------------------------------- 0.3/12.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.4 MB 227.1 kB/s eta 0:00:53
   - -------------------------------------- 0.5/12.4 MB 227.1 kB/s eta 0:00:53
   - -------------------------------------- 0.5/12.4 MB 227.1 kB/s eta 0:00:53
   - -------------------------------------- 0.5/12.4 MB 227.1 kB/s eta 0:00:53
   - -------------------------------------- 0.5/12.4 M

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import tkinter as tk
from tkinter import ttk, messagebox
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict

# ==========================================================
# Color Wars (Your Variant)
# - Board 5x5
# - Start: Human picks a cell with 3 dots, then AI picks a different cell with 3 dots
# - Turn: click ONLY your own cells -> +1 dot
# - If dot count reaches 4: explode/split
#   * origin becomes empty
#   * +1 to orthogonal neighbors, ownership becomes current player (overwrites)
#   * chain reactions possible
# - Win: after both placed, opponent has 0 owned cells
# - Draw: move limit or 3-fold repetition (AI can "at least draw")
# ==========================================================

N = 5
HUMAN = -1
AI = +1

MAX_PLIES = 220
REPEAT_LIMIT = 3

# explosion threshold is constant 4 everywhere in this variant
CRIT = 4


# -----------------------------
# Board (state + explosions)
# -----------------------------
class Board:
    """
    owner[r,c] in {-1, 0, +1} => HUMAN, empty, AI
    dots[r,c]  in {0..3} stable; reaching 4 triggers immediate explosion
    """

    def __init__(self, n=N):
        self.n = n
        self.owner = np.zeros((n, n), dtype=np.int8)
        self.dots = np.zeros((n, n), dtype=np.int8)

    def clone(self) -> "Board":
        b = Board(self.n)
        b.owner = self.owner.copy()
        b.dots = self.dots.copy()
        return b

    def inb(self, r, c) -> bool:
        return 0 <= r < self.n and 0 <= c < self.n

    def neighbors4(self, r, c):
        for dr, dc in ((-1,0),(1,0),(0,-1),(0,1)):
            nr, nc = r + dr, c + dc
            if self.inb(nr, nc):
                yield nr, nc

    def legal_moves(self, player: int) -> List[Tuple[int,int]]:
        # In your game: after placements, you can only click your OWN cells.
        return list(zip(*np.where(self.owner == player)))

    def place_start(self, player: int, r: int, c: int):
        # Starting placement: choose any empty cell, it becomes yours with 3 dots.
        if not self.inb(r, c) or self.owner[r, c] != 0:
            raise ValueError("Illegal start placement")
        self.owner[r, c] = player
        self.dots[r, c] = 3  # fixed start count

    def click(self, player: int, r: int, c: int):
        # During play: can only click your own cells
        if not self.inb(r, c) or self.owner[r, c] != player:
            raise ValueError("Illegal move")
        self.dots[r, c] += 1
        if self.dots[r, c] >= CRIT:
            self._explode_chain(player, [(r, c)])

    def _explode_chain(self, player: int, queue: List[Tuple[int,int]]):
        # process explosions until stable
        while queue:
            r, c = queue.pop()

            # cell might already have been cleared by earlier explosion
            if self.dots[r, c] < CRIT:
                continue

            # explode: clear origin
            self.dots[r, c] = 0
            self.owner[r, c] = 0

            # distribute +1 to neighbors, overwrite ownership to current player
            for nr, nc in self.neighbors4(r, c):
                self.owner[nr, nc] = player
                self.dots[nr, nc] += 1
                if self.dots[nr, nc] >= CRIT:
                    queue.append((nr, nc))

            # keep all stable counts in [0..3] (because >=4 will be exploded by queue)
            # nothing else needed

    def cells_owned(self, player: int) -> int:
        return int((self.owner == player).sum())

    def total_dots(self, player: int) -> int:
        return int(self.dots[self.owner == player].sum())

    def is_terminal(self, ply: int, placements_done: bool) -> Optional[int]:
        """
        Returns:
          AI (+1) if AI wins
          HUMAN (-1) if human wins
          0 for draw (ply limit)
          None if not terminal
        """
        if ply >= MAX_PLIES:
            return 0
        if not placements_done:
            return None

        h_cells = self.cells_owned(HUMAN)
        a_cells = self.cells_owned(AI)

        # If opponent has no cells -> win
        if h_cells == 0 and a_cells > 0:
            return AI
        if a_cells == 0 and h_cells > 0:
            return HUMAN
        return None

    def key(self, turn: int, phase: int) -> bytes:
        """
        phase: 0 = human placement, 1 = ai placement, 2 = play
        """
        return self.owner.tobytes() + self.dots.tobytes() + bytes([turn & 0xFF, phase & 0xFF])


# -----------------------------
# AI: Minimax + Alpha-Beta + Transposition
# -----------------------------
@dataclass
class MinimaxAI:
    max_depth: int = 5

    def __post_init__(self):
        self.tt: Dict[Tuple[bytes, int], float] = {}  # (state_key, depth) -> value

    def choose_move(self, board: Board, ply: int) -> Optional[Tuple[int,int]]:
        moves = board.legal_moves(AI)
        if not moves:
            return None

        # move ordering: prefer corners/edges; prefer capturing enemy 3-cells (danger cells)
        def move_score(rc):
            r, c = rc
            sc = 0
            if (r, c) in [(0,0),(0,N-1),(N-1,0),(N-1,N-1)]:
                sc += 6
            elif r in (0, N-1) or c in (0, N-1):
                sc += 3

            # if any neighbor is enemy with 3, clicking here might cause explosion/capture soon
            for nr, nc in board.neighbors4(r, c):
                if board.owner[nr, nc] == HUMAN and board.dots[nr, nc] == 3:
                    sc += 2
            # prefer higher dot counts (closer to explode)
            sc += int(board.dots[r, c])
            return sc

        moves.sort(key=move_score, reverse=True)

        best_val = -1e18
        best_move = moves[0]
        alpha, beta = -1e18, 1e18

        for (r, c) in moves:
            b2 = board.clone()
            b2.click(AI, r, c)
            val = self._minimax(b2, depth=self.max_depth - 1, alpha=alpha, beta=beta,
                                maximizing=False, ply=ply+1, placements_done=True)
            if val > best_val:
                best_val = val
                best_move = (r, c)
            alpha = max(alpha, best_val)
            if alpha >= beta:
                break

        return best_move

    def _minimax(self, board: Board, depth: int, alpha: float, beta: float,
                 maximizing: bool, ply: int, placements_done: bool) -> float:
        term = board.is_terminal(ply, placements_done)
        if term is not None:
            if term == AI:
                return 1e9 + depth
            if term == HUMAN:
                return -1e9 - depth
            return 0.0  # draw

        if depth <= 0:
            return self.heuristic(board)

        turn = AI if maximizing else HUMAN
        key = board.key(turn=turn, phase=2)
        tt_key = (key, depth)
        if tt_key in self.tt:
            return self.tt[tt_key]

        moves = board.legal_moves(turn)
        if not moves:
            # if somehow no moves, just evaluate
            val = self.heuristic(board)
            self.tt[tt_key] = val
            return val

        # ordering: same idea, helps pruning
        def order_score(rc):
            r, c = rc
            sc = int(board.dots[r, c])
            if (r, c) in [(0,0),(0,N-1),(N-1,0),(N-1,N-1)]:
                sc += 3
            elif r in (0, N-1) or c in (0, N-1):
                sc += 1
            return sc

        moves.sort(key=order_score, reverse=True)

        if maximizing:
            best = -1e18
            for (r, c) in moves:
                b2 = board.clone()
                b2.click(turn, r, c)
                best = max(best, self._minimax(b2, depth-1, alpha, beta, False, ply+1, True))
                alpha = max(alpha, best)
                if alpha >= beta:
                    break
        else:
            best = 1e18
            for (r, c) in moves:
                b2 = board.clone()
                b2.click(turn, r, c)
                best = min(best, self._minimax(b2, depth-1, alpha, beta, True, ply+1, True))
                beta = min(beta, best)
                if alpha >= beta:
                    break

        self.tt[tt_key] = best
        return best

    def heuristic(self, board: Board) -> float:
        """
        Required heuristic (AI perspective):
        1) #cells owned
        2) total #dots
        3) bonus corners/edges
        4) penalty for opponent cells at 3 (critical-1 since crit=4)
        """
        ai_cells = board.cells_owned(AI)
        hu_cells = board.cells_owned(HUMAN)
        ai_dots = board.total_dots(AI)
        hu_dots = board.total_dots(HUMAN)

        corners = [(0,0),(0,N-1),(N-1,0),(N-1,N-1)]
        corner_bonus = sum(1 for r,c in corners if board.owner[r,c] == AI) - \
                       sum(1 for r,c in corners if board.owner[r,c] == HUMAN)

        edge_bonus = 0
        for r in range(N):
            for c in range(N):
                if (r,c) in corners:
                    continue
                if r in (0, N-1) or c in (0, N-1):
                    if board.owner[r,c] == AI:
                        edge_bonus += 1
                    elif board.owner[r,c] == HUMAN:
                        edge_bonus -= 1

        # penalty: opponent cells at 3 are dangerous (one click away from explosion)
        opp_danger = int(((board.owner == HUMAN) & (board.dots == 3)).sum())

        # weights (tune anytime)
        return (
            10.0 * (ai_cells - hu_cells)
            + 4.0 * (ai_dots - hu_dots)
            + 10.0 * corner_bonus
            + 3.0 * edge_bonus
            - 9.0 * opp_danger
        )


# -----------------------------
# Tkinter GUI
# -----------------------------
class ColorWarsGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Color Wars (5x5) — Your Rules — Human vs AI (Minimax)")

        top = tk.Frame(root)
        top.pack(fill="x", padx=10, pady=8)

        tk.Label(top, text="AI Depth:").pack(side="left")
        self.depth_var = tk.IntVar(value=5)
        self.depth_scale = ttk.Scale(top, from_=2, to=7, orient="horizontal",
                                     length=180)
        self.depth_scale.set(5)
        self.depth_scale.pack(side="left", padx=8)

        self.depth_label = tk.Label(top, text="5")
        self.depth_label.pack(side="left")
        self.depth_scale.configure(command=self._depth_changed)

        tk.Button(top, text="New Game", command=self.new_game).pack(side="right")

        self.status = tk.StringVar(value="")
        tk.Label(root, textvariable=self.status, fg="blue").pack(pady=(0, 6))

        self.cell = 90
        self.pad = 20
        w = self.pad*2 + self.cell*N
        h = self.pad*2 + self.cell*N
        self.canvas = tk.Canvas(root, width=w, height=h, bg="white")
        self.canvas.pack(padx=10, pady=10)
        self.canvas.bind("<Button-1>", self.on_click)

        self.new_game()

    def _depth_changed(self, v):
        d = int(round(float(v)))
        d = max(2, min(7, d))
        self.depth_var.set(d)
        self.depth_label.config(text=str(d))

    def new_game(self):
        self.board = Board(N)
        self.ai = MinimaxAI(max_depth=self.depth_var.get())

        self.phase = 0  # 0 human place, 1 ai place, 2 play
        self.turn = HUMAN
        self.ply = 0

        self.rep: Dict[bytes, int] = {}  # for draw by repetition

        self.status.set("Phase 1: Click ANY empty cell to place your start (3 dots).")
        self.draw()

    def touch_rep(self):
        k = self.board.key(turn=self.turn, phase=self.phase)
        self.rep[k] = self.rep.get(k, 0) + 1
        return self.rep[k]

    def check_draw_or_terminal(self) -> Optional[int]:
        # draw by repetition
        if self.touch_rep() >= REPEAT_LIMIT:
            return 0
        # draw by move limit or win/lose
        placements_done = (self.phase == 2)
        term = self.board.is_terminal(self.ply, placements_done)
        return term

    def finish(self, res: int):
        if res == 0:
            messagebox.showinfo("Game Over", "Draw!")
        elif res == AI:
            messagebox.showinfo("Game Over", "AI wins! 💀")
        else:
            messagebox.showinfo("Game Over", "You win! 🎉")

    def on_click(self, event):
        x, y = event.x, event.y
        c = int((x - self.pad) // self.cell)
        r = int((y - self.pad) // self.cell)
        if not self.board.inb(r, c):
            return

        # --- Placement phase: Human picks start ---
        if self.phase == 0:
            if self.board.owner[r, c] != 0:
                self.status.set("Pick an EMPTY cell for start placement.")
                return
            self.board.place_start(HUMAN, r, c)
            self.ply += 1
            self.phase = 1
            self.draw()
            self.status.set("AI placing start...")
            self.root.after(120, self.ai_place_start)
            return

        # --- During play ---
        if self.phase != 2:
            return
        if self.turn != HUMAN:
            return

        if self.board.owner[r, c] != HUMAN:
            self.status.set("Illegal: you can click ONLY your RED cells.")
            return

        # Human move
        self.board.click(HUMAN, r, c)
        self.ply += 1
        self.draw()

        term = self.check_draw_or_terminal()
        if term is not None:
            self.finish(term)
            return

        # AI move
        self.turn = AI
        self.status.set("AI thinking...")
        self.root.after(100, self.ai_turn)

    def ai_place_start(self):
        # AI chooses a random empty cell for start placement (3 dots)
        empties = list(zip(*np.where(self.board.owner == 0)))
        if not empties:
            self.finish(0)
            return
        r, c = empties[np.random.randint(len(empties))]
        self.board.place_start(AI, r, c)
        self.ply += 1

        # start main play
        self.phase = 2
        self.turn = HUMAN
        self.rep = {}  # start repetition counting from here
        self.touch_rep()

        self.draw()
        self.status.set("Phase 2: Your turn (RED). Click your own cell to +1 (explode at 4).")

    def ai_turn(self):
        if self.phase != 2 or self.turn != AI:
            return

        # update depth live
        self.ai.max_depth = self.depth_var.get()
        self.ai.tt.clear()  # clear TT when depth changes; safe + prevents stale eval

        mv = self.ai.choose_move(self.board, self.ply)
        if mv is None:
            self.finish(0)
            return

        r, c = mv
        self.board.click(AI, r, c)
        self.ply += 1
        self.draw()

        term = self.check_draw_or_terminal()
        if term is not None:
            self.finish(term)
            return

        self.turn = HUMAN
        self.status.set("Your turn (RED). Click your own cell.")

    # -----------------------------
    # Drawing (circles like your screenshots)
    # -----------------------------
    def draw(self):
        self.canvas.delete("all")

        # background + cells
        for r in range(N):
            for c in range(N):
                x1 = self.pad + c*self.cell
                y1 = self.pad + r*self.cell
                x2 = x1 + self.cell
                y2 = y1 + self.cell

                # base tile
                base = "#f5eddc"
                self.canvas.create_rectangle(x1, y1, x2, y2, fill=base, outline="#d7cbb8")

                own = self.board.owner[r, c]
                d = int(self.board.dots[r, c])

                if d > 0:
                    # circle background color
                    if own == HUMAN:
                        circ = "#ff5b5b"
                    elif own == AI:
                        circ = "#1db6ff"
                    else:
                        circ = "#cccccc"

                    # big disk
                    cx, cy = (x1+x2)/2, (y1+y2)/2
                    rad = self.cell * 0.33
                    self.canvas.create_oval(cx-rad, cy-rad, cx+rad, cy+rad, fill=circ, outline="")

                    # draw 1..3 small white dots inside (like screenshot)
                    self._draw_pips(cx, cy, d)

        # text
        if self.phase == 2:
            # show score (cells owned)
            h = self.board.cells_owned(HUMAN)
            a = self.board.cells_owned(AI)
            self.canvas.create_text(self.pad, self.pad-10, anchor="w",
                                    text=f"RED(Human) cells: {h}   BLUE(AI) cells: {a}   Ply: {self.ply}",
                                    fill="#333333", font=("Arial", 10, "bold"))

    def _draw_pips(self, cx, cy, d):
        # pip positions for 1,2,3
        off = self.cell * 0.10
        r = self.cell * 0.04

        if d == 1:
            pts = [(cx, cy)]
        elif d == 2:
            pts = [(cx-off, cy), (cx+off, cy)]
        else:  # d == 3
            pts = [(cx-off, cy-off*0.8), (cx+off, cy-off*0.8), (cx, cy+off*0.9)]

        for (x, y) in pts:
            self.canvas.create_oval(x-r, y-r, x+r, y+r, fill="white", outline="")


# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    root = tk.Tk()
    app = ColorWarsGUI(root)
    root.mainloop()
